What I have practiced:

NumPy: vectorization, broadcasting, numerical stability  
Pandas: joins, groupby, apply vs vectorized, pivot, clean columns  
OOP: classes, composition, dataclasses (optional), design separation  
Finance: Black-Scholes, Greeks, scenario risk  

Part A — Dataset (15 minutes)   
Create a small DataFrame in code (no files needed). Use exactly this schema:  
Trades table  
Each row is one option trade.  
Columns:  
trade_id (int)  
underlying (str) e.g. "AAPL"  
option_type (str) "call" or "put"  
K (float) strike  
T (float) time to expiry in years (e.g. 0.25)  
qty (int) positive = long, negative = short  
multiplier (int) equity option contract multiplier (usually 100)  

In [1]:
import pandas as pd
import numpy as np
# !pip install scipy
from scipy.stats import norm

In [2]:
# Trades table
trades_data = pd.DataFrame(
    data=[
    (1, "AAPL", "call", 190, 0.25,  10, 100),
    (2, "AAPL", "put",  180, 0.25,  -5, 100),
    (3, "MSFT", "call", 420, 0.50,   7, 100),
    (4, "MSFT", "put",  400, 0.50,   4, 100),
    (5, "NVDA", "call", 900, 0.10,  -8, 100),
    ], 
    columns=[
        'trade_id', 
        'underlying', 
        'option_type',
        'K',
        'T',
        'qty',
        'multiplier'
    ]
)

trades_data

,trade_id,underlying,option_type,K,T,qty,multiplier
0,1,AAPL,call,190,0.25,10,100
1,2,AAPL,put,180,0.25,-5,100
2,3,MSFT,call,420,0.50,7,100
3,4,MSFT,put,400,0.50,4,100
4,5,NVDA,call,900,0.10,-8,100


In [3]:
# Market data table
mkt_data = pd.DataFrame(
    data=[
    ("AAPL", 185.0, 0.28, 0.05),
    ("MSFT", 410.0, 0.22, 0.05),
    ("NVDA", 880.0, 0.45, 0.05),
    ],
    columns=[
        'underlying',
        'S',
        'sigma',
        'r'
    ]
)
mkt_data

,underlying,S,sigma,r
0,AAPL,185.0,0.28,0.05
1,MSFT,410.0,0.22,0.05
2,NVDA,880.0,0.45,0.05


Merge them into one DataFrame book_df on underlying  
Checkpoint: book_df should have S, sigma, r on each trade row.

In [4]:
book_df = pd.merge(trades_data, mkt_data, on='underlying')
book_df

,trade_id,underlying,option_type,K,T,qty,multiplier,S,sigma,r
0,1,AAPL,call,190,0.25,10,100,185.0,0.28,0.05
1,2,AAPL,put,180,0.25,-5,100,185.0,0.28,0.05
2,3,MSFT,call,420,0.50,7,100,410.0,0.22,0.05
3,4,MSFT,put,400,0.50,4,100,410.0,0.22,0.05
4,5,NVDA,call,900,0.10,-8,100,880.0,0.45,0.05


Part B — NumPy Black-Scholes

### Black-Scholes: d1 and d2

The two standard intermediate quantities are:

- `d1 = (log(S/K) + (r + 0.5*sigma^2)*T) / (sigma*sqrt(T))`
- `d2 = d1 - sigma*sqrt(T)`

I implement them as standalone functions that accept both scalars and NumPy arrays, so I can apply them directly to DataFrame columns without any explicit loop.

### Black-Scholes call and put prices

The closed-form prices are:

- **Call:** `C = S * N(d1) - K * exp(-rT) * N(d2)`
- **Put:**  `P = K * exp(-rT) * N(-d2) - S * N(-d1)`

I use `np.where` to handle both option types in a single vectorised expression rather than branching on `option_type` with an if-else. This keeps the function concise and avoids Python-level row iteration.

### Greeks: Delta, Gamma, Vega

| Greek | Call formula | Put formula |
|-------|-------------|-------------|
| Delta | N(d1) | N(d1) - 1 |
| Gamma | phi(d1) / (S * sigma * sqrt(T)) | same |
| Vega  | S * phi(d1) * sqrt(T) | same |

I use `norm.pdf` (the standard normal PDF, phi) for Gamma and Vega — not `norm.cdf`. Vega here is expressed per unit of volatility, so a 1 vol-point shock corresponds to multiplying vega by 0.01.

In [5]:
# Use np.log, np.exp, np.sqrt, np.maximum
# Inputs are arrays not numbers

# Option d1, d2
def bs_d1(S, K, r, sigma, T, option_type):
    return (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))

def bs_d2(S, K, r, sigma, T, option_type):
    return (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T)) - sigma * np.sqrt(T)

# Option price
def bs_price(S, K, r, sigma, T, option_type):
    d1 = bs_d1(S, K, r, sigma, T, option_type)
    d2 = bs_d2(S, K, r, sigma, T, option_type)
    return np.where(
        option_type == 'call',
        S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2),
        K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    )

def bs_delta(S, K, r, sigma, T, option_type):
    d1 = bs_d1(S, K, r, sigma, T, option_type)
    d2 = bs_d2(S, K, r, sigma, T, option_type)
    return np.where(
        option_type == 'call',
        norm.cdf(d1),
        norm.cdf(d1) - 1
    )

def bs_gamma(S, K, r, sigma, T, option_type):
    d1 = bs_d1(S, K, r, sigma, T, option_type)
    return (norm.pdf(d1)/(S * sigma * np.sqrt(T)))
    # return (norm.cdf(d1)/(S * sigma * np.sqrt(T)))

def bs_vega(S, K, r, sigma, T, option_type):
    # Per 1.00 vol, not per 1% - be consistent and document
    d1 = bs_d1(S, K, r, sigma, T, option_type)
    return (S * norm.pdf(d1) * np.sqrt(T))
    # return (S * norm.cdf(d1) * np.sqrt(T))

In [6]:
# Add columns to book
S = book_df['S']
K = book_df['K']
r = book_df['r']
sigma = book_df['sigma']
T = book_df['T']
option_type = book_df['option_type']

book_df['price'] = bs_price(S, K, r, sigma, T, option_type)
book_df['delta'] = bs_delta(S, K, r, sigma, T, option_type)
book_df['gamma'] = bs_gamma(S, K, r, sigma, T, option_type)
book_df['vega'] = bs_vega(S, K, r, sigma, T, option_type)

book_df

,trade_id,underlying,option_type,K,T,qty,multiplier,S,sigma,r,price,delta,gamma,vega
0,1,AAPL,call,190,0.25,10,100,185.0,0.28,0.05,9.131053,0.487554,0.015396,36.884202
1,2,AAPL,put,180,0.25,-5,100,185.0,0.28,0.05,6.911843,-0.361298,0.014463,34.648700
2,3,MSFT,call,420,0.50,7,100,410.0,0.22,0.05,25.593197,0.533306,0.006233,115.255567
3,4,MSFT,put,400,0.50,4,100,410.0,0.22,0.05,16.117108,-0.345603,0.005780,106.885053
4,5,NVDA,call,900,0.10,-8,100,880.0,0.45,0.05,42.978676,0.479410,0.003182,110.869931


Part C - OOP Structure

## Part C — OOP Structure

### `EuropeanOption`

I model each contract as an `EuropeanOption` that stores its defining terms (`option_type`, `K`, `T`, `multiplier`) and exposes `price(S, r, sigma)` and `greeks(S, r, sigma)` methods that delegate to the vectorised BS functions above. Keeping the option stateless with respect to market data means I can reprice the same object under any scenario without any mutation.

### `Position`

A `Position` wraps an `EuropeanOption` with a signed `qty` and a string `underlying` label. The `value` method scales option price by `qty * multiplier`, and `risk` scales the Greeks vector by the same factor. This is the natural unit of a trading book: a signed quantity of a specific contract.

### `Book`

`Book` is a container of `Position` objects with a few key methods:

- `from_dataframe` — a class method that builds the book from a raw trades DataFrame, so the caller doesn't need to construct `EuropeanOption` and `Position` objects manually.
- `run_risk` — iterates over positions, looks up the market data for each underlying, and returns a trade-level risk table with price, PV, and scaled Greeks.
- `aggregate_by_underlying` — a static helper that groups the risk table by underlying to give a net Greeks view per name.

In [7]:
class EuropeanOption:
    def __init__(self, option_type, K, T, multiplier):
        self.option_type = option_type
        self.K = K
        self.T = T
        self.multiplier = multiplier

    # Methods
    def price(self, S, r, sigma): # Vectorized allowed
        return bs_price(S, self.K, r, sigma, self.T, self.option_type)
    
    def greeks(self, S, r, sigma):
        S = np.asarray(S)
        G = np.column_stack([
            bs_delta(S, self.K, r, sigma, self.T, self.option_type),
            bs_gamma(S, self.K, r, sigma, self.T, self.option_type),
            bs_vega(S, self.K, r, sigma, self.T, self.option_type)
        ])

        if G.shape[0] == 1 and S.ndim == 0:        # handle scalar nicely (optional)
            return G.ravel()                       # shape (3,)
        
        return G
    

class Position:
    # Holds instrument (European option), qty, underlying
    def __init__(self, instrument: EuropeanOption, qty, underlying):
        self.instrument = instrument
        self.qty = qty
        self.underlying = underlying    # A string

    def value(self, S, r, sigma):   # Position PV = qty * multiplier * option_price
        return (self.instrument.price(S, r, sigma) * self.qty * self.instrument.multiplier)
    
    def risk(self, S, r, sigma):    # Positional greeks scaled by qty * multiplier
        scale = self.qty * self.instrument.multiplier
        return self.instrument.greeks(S, r, sigma) * scale
    

class Book:
    def __init__(self, positions, trades_df=None):
        """
        positions: list[Position]
        trades_df: optional - original trades (for trade_id, K, T, option_type, etc.)
        """
        self.positions = positions
        self.trades_df = trades_df

    @classmethod
    def from_dataframe(cls, trades_df: pd.DataFrame):
        """
        Build a Book from trades_df with columns:
        trade_id, underlying, option_type, K, T, qty, multiplier
        """
        positions = []
        for row in trades_df.itertuples(index=False):
            opt = EuropeanOption(
                option_type=row.option_type,
                K=row.K,
                T=row.T,
                multiplier=row.multiplier
            )
            pos = Position(
                instrument=opt,
                qty=row.qty,
                underlying=row.underlying
            )
            positions.append(pos)

        return cls(positions=positions, trades_df=trades_df.copy())
    
    def run_risk(self, mkt_df: pd.DataFrame) -> pd.DataFrame:
        """
        Returns a trade-level risk DataFrame with:
        trade_id, underlying, qty, multiplier, price, pv, delta, gamma, vega
        (delta/gamma/vega are already $-scaled by qty*multiplier, but NOT spot-scaled)
        """
        # Map underlying -> (S, r, sigma)
        mkt_map = (
            mkt_df.set_index("underlying")[["S", "r", "sigma"]]
            .to_dict(orient="index")
        )

        rows = []
        for i, pos in enumerate(self.positions):
            if pos.underlying not in mkt_map:
                raise KeyError(f"Missing market data for underlying={pos.underlying}")

            S = mkt_map[pos.underlying]["S"]
            r = mkt_map[pos.underlying]["r"]
            sigma = mkt_map[pos.underlying]["sigma"]

            # Option price per share
            price = pos.instrument.price(S, r, sigma)

            # Position PV (qty * multiplier * price)
            pv = pos.value(S, r, sigma)

            # Scaled greeks (qty*multiplier)
            d, g, v = pos.risk(S, r, sigma)

            trade_id = None
            if self.trades_df is not None and "trade_id" in self.trades_df.columns:
                # assume same order as positions were created
                trade_id = self.trades_df.iloc[i]["trade_id"]

            rows.append({
                "trade_id": trade_id,
                "underlying": pos.underlying,
                "qty": pos.qty,
                "multiplier": pos.instrument.multiplier,
                "option_type": pos.instrument.option_type,
                "K": pos.instrument.K,
                "T": pos.instrument.T,
                "price": float(price),
                "pv": float(pv),
                "delta": float(d),
                "gamma": float(g),
                "vega": float(v),
            })

        return pd.DataFrame(rows)

    @staticmethod
    def aggregate_by_underlying(risk_df: pd.DataFrame) -> pd.DataFrame:
        """
        Group and sum PV and greeks by underlying.
        """
        agg = (risk_df
               .groupby("underlying", as_index=False)[["pv", "delta", "gamma", "vega"]]
               .sum())
        return agg

In [8]:
book = Book.from_dataframe(trades_data)
# Trade level risk
risk_df = book.run_risk(mkt_data)
risk_df

,trade_id,underlying,qty,multiplier,option_type,K,T,price,pv,delta,gamma,vega
0,1,AAPL,10,100,call,190,0.25,9.131053,9131.053102,487.554316,15.395681,36884.202257
1,2,AAPL,-5,100,put,180,0.25,6.911843,-3455.921658,180.648770,-7.231285,-17324.350096
2,3,MSFT,7,100,call,420,0.50,25.593197,17915.237899,373.314179,4.363144,80678.896972
3,4,MSFT,4,100,put,400,0.50,16.117108,6446.843303,-138.241348,2.312153,42754.021076
4,5,NVDA,-8,100,call,900,0.10,42.978676,-34382.940951,-383.527663,-2.545223,-88695.944408


In [9]:
# Risks grouped by underlying
agg_df = book.aggregate_by_underlying(risk_df)
agg_df

,underlying,pv,delta,gamma,vega
0,AAPL,5675.131444,668.203086,8.164396,19559.852160
1,MSFT,24362.081202,235.072831,6.675297,123432.918048
2,NVDA,-34382.940951,-383.527663,-2.545223,-88695.944408


Part D — Scenario Shocks & PnL Explain

In [10]:
# Helper function
def book_pv(book: Book, mkt_df):
    scn_risk = book.run_risk(mkt_df)
    return scn_risk["pv"].sum()

In [11]:
# Create simple scenarios (spot ±1%, vol ±2 vol points)
# Note: your vega is per 1.00 vol (e.g. +1.00), so 2 vol points = 0.02.
def make_scenarios(mkt_df):
    base = mkt_df.copy()
    scenarios = {"base": base}

    # Spot shocks
    upS = base.copy()
    upS["S"] = upS["S"] * 1.01
    scenarios["spot_up_1pct"] = upS

    dnS = base.copy()
    dnS["S"] = dnS["S"] * 0.99
    scenarios["spot_down_1pct"] = dnS

    # Vol shocks
    upV = base.copy()
    upV["sigma"] = upV["sigma"] + 0.02
    scenarios["vol_up_2pt"] = upV

    dnV = base.copy()
    dnV["sigma"] = np.maximum(dnV["sigma"] - 0.02, 0.01)  # floor vol
    scenarios["vol_down_2pt"] = dnV

    return scenarios

scenarios = make_scenarios(mkt_data)
scenarios

{'base':   underlying      S  sigma     r
 0       AAPL  185.0   0.28  0.05
 1       MSFT  410.0   0.22  0.05
 2       NVDA  880.0   0.45  0.05,
 'spot_up_1pct':   underlying       S  sigma     r
 0       AAPL  186.85   0.28  0.05
 1       MSFT  414.10   0.22  0.05
 2       NVDA  888.80   0.45  0.05,
 'spot_down_1pct':   underlying       S  sigma     r
 0       AAPL  183.15   0.28  0.05
 1       MSFT  405.90   0.22  0.05
 2       NVDA  871.20   0.45  0.05,
 'vol_up_2pt':   underlying      S  sigma     r
 0       AAPL  185.0   0.30  0.05
 1       MSFT  410.0   0.24  0.05
 2       NVDA  880.0   0.47  0.05,
 'vol_down_2pt':   underlying      S  sigma     r
 0       AAPL  185.0   0.26  0.05
 1       MSFT  410.0   0.20  0.05
 2       NVDA  880.0   0.43  0.05}

In [12]:
# Full repricing PnL table
base_pv = book_pv(book, scenarios["base"])

rows = []
for name, mkt_scn in scenarios.items():
    pv = book_pv(book, mkt_scn)
    rows.append({"scenario": name, "book_pv": pv, "pnl_vs_base": pv - base_pv})


scn_full = pd.DataFrame(rows).sort_values("scenario").reset_index(drop=True)
scn_full

,scenario,book_pv,pnl_vs_base
0,base,-4345.728306,0.000000
1,spot_down_1pct,-3198.984878,1146.743428
2,spot_up_1pct,-5549.385661,-1203.657355
3,vol_down_2pt,-5429.253433,-1083.525128
4,vol_up_2pt,-3257.968921,1087.759385


In [13]:
# Greeks-approx PnL (compare vs full repricing)
# Base greeks summed by underlying (position-scaled already)
base_greeks = risk_df.groupby("underlying")[["delta", "gamma", "vega"]].sum()

# Base market indexed by underlying
base_mkt = mkt_data.set_index("underlying")[["S", "sigma"]]

def approx_pnl_for_scenario(name):
    # default shocks
    dS = pd.Series(0.0, index=base_greeks.index)
    dSigma = pd.Series(0.0, index=base_greeks.index)

    if name == "spot_up_1pct":
        dS = base_mkt["S"] * 0.01
    elif name == "spot_down_1pct":
        dS = base_mkt["S"] * -0.01
    elif name == "vol_up_2pt":
        dSigma = pd.Series(0.02, index=base_greeks.index)
    elif name == "vol_down_2pt":
        dSigma = pd.Series(-0.02, index=base_greeks.index)

    # Δ + 0.5ΓΔS^2 + Vega*Δσ
    approx = (base_greeks["delta"] * dS
              + 0.5 * base_greeks["gamma"] * (dS ** 2)
              + base_greeks["vega"] * dSigma).sum()

    return approx

# attach approx pnl to your full repricing table
scn_full["pnl_approx"] = scn_full["scenario"].apply(approx_pnl_for_scenario)
scn_full["approx_error"] = scn_full["pnl_vs_base"] - scn_full["pnl_approx"]

scn_full

,scenario,book_pv,pnl_vs_base,pnl_approx,approx_error
0,base,-4345.728306,0.000000,0.000000,0.000000
1,spot_down_1pct,-3198.984878,1146.743428,1146.595265,0.148163
2,spot_up_1pct,-5549.385661,-1203.657355,-1203.542974,-0.114381
3,vol_down_2pt,-5429.253433,-1083.525128,-1085.936516,2.411388
4,vol_up_2pt,-3257.968921,1087.759385,1085.936516,1.822869
